# M4: Feature Engineering & Baseline Model

**Author:** Venkata Kanna Bhavan Surya Adapa (CB.SC.U4CSE23467)

**Issues:** 11, 12, 13

This notebook implements:
- **Issue 11:** Structured Feature Engineering
- **Issue 12:** Text Feature Engineering  
- **Issue 13:** Define Prediction Target & Baseline

In [ ]:
import pandas as pd
import numpy as np
import ast
import re
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

## Load Data

In [ ]:
# Load cleaned dataset
df = pd.read_csv("../data/processed/tmdb_clean.csv", parse_dates=["release_date"])
df["genres"] = df["genres"].apply(ast.literal_eval)
print(f"Loaded {len(df):,} movies")
print(f"Columns: {list(df.columns)}")

---
# Issue 11: Structured Feature Engineering

Create useful features from structured movie data.

## 11.1 Extract Release Year

In [ ]:
# Release year already exists, verify it
print(f"Release year range: {df['release_year'].min()} - {df['release_year'].max()}")
print(f"Missing release_year: {df['release_year'].isna().sum()}")

## 11.2 Extract Release Month

In [ ]:
# Extract release month
df["release_month"] = df["release_date"].dt.month
print(f"Release month distribution:\n{df['release_month'].value_counts().sort_index()}")

## 11.3 Create Decade Feature

In [ ]:
# Create decade feature
df["decade"] = (df["release_year"] // 10) * 10
print(f"Decade distribution:\n{df['decade'].value_counts().sort_index()}")

## 11.4 Encode Genres (Multi-label)

In [ ]:
# Encode genres using MultiLabelBinarizer
mlb = MultiLabelBinarizer()
genre_encoded = pd.DataFrame(
    mlb.fit_transform(df["genres"]),
    index=df.index,
    columns=[f"g_{g}" for g in mlb.classes_]
)
print(f"Genre features created: {len(mlb.classes_)}")
print(f"Genres: {list(mlb.classes_)}")

## 11.5 Encode Languages

In [ ]:
# Encode top 10 languages + 'other'
top_languages = df["original_language"].value_counts().head(10).index.tolist()
df["language"] = df["original_language"].where(
    df["original_language"].isin(top_languages), "other"
)
language_encoded = pd.get_dummies(df["language"], prefix="l", dtype=int)
print(f"Language features created: {len(language_encoded.columns)}")
print(f"Languages: {list(language_encoded.columns)}")

## 11.6 Prepare Model-Ready Features

In [ ]:
# Filter to movies with 20+ votes (reliable ratings)
model_df = df[df["vote_count"] >= 20].copy()
print(f"Movies for modelling: {len(model_df):,} (with 20+ votes)")

# Additional features
model_df["overview_words"] = model_df["overview"].str.split().str.len()
model_df["n_genres"] = model_df["genres"].str.len()
model_df["log_popularity"] = np.log1p(model_df["popularity"])
model_df["log_votes"] = np.log1p(model_df["vote_count"])

# Recreate encodings for filtered data
genre_features = pd.DataFrame(
    mlb.transform(model_df["genres"]),
    index=model_df.index,
    columns=[f"g_{g}" for g in mlb.classes_]
)

model_df["language"] = model_df["original_language"].where(
    model_df["original_language"].isin(top_languages), "other"
)
lang_features = pd.get_dummies(model_df["language"], prefix="l", dtype=int)

# Feature Set A: Content only (pre-release)
base_features = ["release_year", "release_month", "overview_words", "n_genres"]
X_A = pd.concat([model_df[base_features], genre_features, lang_features], axis=1)

# Feature Set B: With engagement (post-release)
X_B = pd.concat([X_A, model_df[["log_popularity", "log_votes"]]], axis=1)

print(f"\nFeature Set A (content only): {X_A.shape[1]} features")
print(f"Feature Set B (with engagement): {X_B.shape[1]} features")

## 11.7 Validate Features

In [ ]:
print("=== Feature Validation ===")
print(f"Rows: {X_B.shape[0]:,}")
print(f"Columns: {X_B.shape[1]}")
print(f"Missing values: {X_B.isna().sum().sum()}")
print(f"Duplicate rows: {X_B.duplicated().sum()}")
print(f"Infinite values: {np.isinf(X_B.select_dtypes(include=[np.number])).sum().sum()}")
print(f"\nData types:\n{X_B.dtypes.value_counts()}")

---
# Issue 12: Text Feature Engineering

Prepare movie overview text for text mining and modelling.

## 12.1 Handle Missing Text

In [ ]:
# Check missing/empty overviews
missing_overview = model_df["overview"].isna().sum()
empty_overview = (model_df["overview"].str.strip() == "").sum()
print(f"Missing overviews: {missing_overview}")
print(f"Empty overviews: {empty_overview}")

## 12.2 Clean Overview Text

In [ ]:
def clean_text(text):
    """Clean text for TF-IDF processing."""
    if pd.isna(text) or text.strip() == "":
        return ""
    # Lowercase
    text = text.lower()
    # Remove non-alphabetic characters
    text = re.sub(r"[^a-z\s]", " ", text)
    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text

model_df["clean_overview"] = model_df["overview"].apply(clean_text)
print(f"Sample cleaned text:\n{model_df['clean_overview'].iloc[0][:200]}...")

## 12.3 Generate TF-IDF Features

In [ ]:
# TF-IDF configuration
TFIDF_CONFIG = {
    "stop_words": "english",
    "max_features": 1000,
    "ngram_range": (1, 2),
    "min_df": 5
}

print("TF-IDF Configuration:")
for k, v in TFIDF_CONFIG.items():
    print(f"  {k}: {v}")

## 12.4 Identify Important Terms

In [ ]:
# Fit TF-IDF on all data for term analysis (will refit on train only for modelling)
tfidf_analyzer = TfidfVectorizer(**TFIDF_CONFIG)
tfidf_matrix = tfidf_analyzer.fit_transform(model_df["clean_overview"])

# Get feature names and average TF-IDF scores
feature_names = tfidf_analyzer.get_feature_names_out()
avg_tfidf = np.asarray(tfidf_matrix.mean(axis=0)).ravel()
top_terms = pd.Series(avg_tfidf, index=feature_names).sort_values(ascending=False)

print(f"Vocabulary size: {len(feature_names)}")
print(f"\nTop 15 terms by average TF-IDF:")
print(top_terms.head(15))

## 12.5 Validate Text Features

In [ ]:
print("=== Text Feature Validation ===")
print(f"Documents: {tfidf_matrix.shape[0]:,}")
print(f"Vocabulary terms: {tfidf_matrix.shape[1]:,}")
print(f"Matrix shape: {tfidf_matrix.shape}")
sparsity = 1 - (tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1]))
print(f"Sparsity: {sparsity:.2%}")

---
# Issue 13: Define Prediction Target & Baseline

Define movie performance prediction target and establish baseline model.

## 13.1 Define Prediction Target

In [ ]:
# Target: high_rated = vote_average >= 6.5
# As defined in README.md
TARGET_THRESHOLD = 6.5

model_df["high_rated"] = (model_df["vote_average"] >= TARGET_THRESHOLD).astype(int)
y = model_df["high_rated"]

print("=== Prediction Target ===")
print(f"Target: high_rated (vote_average >= {TARGET_THRESHOLD})")
print(f"Type: Binary Classification")
print(f"\nClass distribution:")
print(f"  Not high-rated (0): {(y == 0).sum():,} ({(y == 0).mean():.1%})")
print(f"  High-rated (1): {(y == 1).sum():,} ({(y == 1).mean():.1%})")

## 13.2 Train/Test Split

In [ ]:
# Stratified train/test split
RANDOM_STATE = 42
TEST_SIZE = 0.2

X_train, X_test, y_train, y_test = train_test_split(
    X_B, y, 
    test_size=TEST_SIZE, 
    stratify=y, 
    random_state=RANDOM_STATE
)

print("=== Train/Test Split ===")
print(f"Random state: {RANDOM_STATE}")
print(f"Test size: {TEST_SIZE}")
print(f"\nTrain: {len(X_train):,} samples ({y_train.mean():.1%} positive)")
print(f"Test: {len(X_test):,} samples ({y_test.mean():.1%} positive)")

## 13.3 TF-IDF on Train Only (No Leakage)

In [ ]:
# Fit TF-IDF on training data only
train_idx = X_train.index
test_idx = X_test.index

tfidf = TfidfVectorizer(**TFIDF_CONFIG)
T_train = tfidf.fit_transform(model_df.loc[train_idx, "clean_overview"])
T_test = tfidf.transform(model_df.loc[test_idx, "clean_overview"])

print("TF-IDF fitted on training data only (no leakage)")
print(f"Train TF-IDF: {T_train.shape}")
print(f"Test TF-IDF: {T_test.shape}")

## 13.4 Baseline Model

In [ ]:
# Baseline: Dummy classifier (majority class)
baseline = DummyClassifier(strategy="prior", random_state=RANDOM_STATE)
baseline.fit(X_train, y_train)

# Predictions
y_pred = baseline.predict(X_test)
y_prob = baseline.predict_proba(X_test)[:, 1]

print("Baseline Model: DummyClassifier (majority class)")

## 13.5 Baseline Metrics

In [ ]:
# Calculate metrics
baseline_results = {
    "Model": "DummyClassifier (baseline)",
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1": f1_score(y_test, y_pred, zero_division=0),
    "AUC": roc_auc_score(y_test, y_prob)
}

print("=== Baseline Results ===")
for metric, value in baseline_results.items():
    if metric != "Model":
        print(f"{metric}: {value:.3f}")

# Save results
results_df = pd.DataFrame([baseline_results])
results_df.to_csv("../data/processed/baseline_results.csv", index=False)
print("\nResults saved to: data/processed/baseline_results.csv")

---
# Summary

## Issue 11: Structured Feature Engineering
- Extracted: release_year, release_month, decade
- Encoded: 19 genre features, 11 language features
- Created: Feature Set A (34) and Set B (36)

## Issue 12: Text Feature Engineering  
- Cleaned overview text (lowercase, remove non-alpha)
- TF-IDF: 1,000 features, ngrams (1,2), min_df=5
- Fitted on training data only

## Issue 13: Prediction Target & Baseline
- Target: high_rated (vote_average >= 6.5)
- Split: 80/20 stratified, random_state=42
- Baseline: DummyClassifier (Accuracy: 0.610, AUC: 0.500)

In [ ]:
print("M4 Feature Engineering Complete!")